In [1]:
import pandas as pd
import sys
sys.path.append('C:/Users/diyad/Desktop/HireSafe/src')

from signal_extraction.signals import SignalExtractor

# Load your preprocessed data
train_df = pd.read_csv('C:/Users/diyad/Desktop/HireSafe/data/train.csv')

print(f"Loaded {len(train_df)} training samples")
print(f"Columns: {train_df.columns.tolist()}")

Loaded 14304 training samples
Columns: ['title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent', 'in_balanced_dataset', 'has_company_profile', 'full_text']


In [2]:
# Initialize extractor
extractor = SignalExtractor()

# Extract signals for all rows
print("Extracting signals from training data...")

signals_list = []
for idx, row in train_df.iterrows():
    signals = extractor.extract_all_signals(row)
    signals_list.append(signals)
    
    if idx % 1000 == 0:
        print(f"Processed {idx}/{len(train_df)} rows...")

# Convert to DataFrame
signals_df = pd.DataFrame(signals_list)

print("\n✓ Signal extraction complete!")
print(f"\nSignal statistics:")
print(signals_df.sum())

Extracting signals from training data...
Processed 0/14304 rows...
Processed 1000/14304 rows...
Processed 2000/14304 rows...
Processed 3000/14304 rows...
Processed 4000/14304 rows...
Processed 5000/14304 rows...
Processed 6000/14304 rows...
Processed 7000/14304 rows...
Processed 8000/14304 rows...
Processed 9000/14304 rows...
Processed 10000/14304 rows...
Processed 11000/14304 rows...
Processed 12000/14304 rows...
Processed 13000/14304 rows...
Processed 14000/14304 rows...

✓ Signal extraction complete!

Signal statistics:
payment_request        3211
urgency                1387
offplatform_contact     356
vague_company          2687
salary_anomaly           56
dtype: int64


In [3]:
# Add signals to training data
train_with_signals = pd.concat([train_df, signals_df], axis=1)

# Check correlation with fraudulent label
print("Signal correlation with fraud label:")
print("\n" + "="*60)

for signal in signals_df.columns:
    fraud_rate = train_with_signals[train_with_signals[signal] == 1]['fraudulent'].mean()
    total_with_signal = train_with_signals[signal].sum()
    
    print(f"\n{signal}:")
    print(f"  - Found in {total_with_signal} postings ({total_with_signal/len(train_df)*100:.1f}%)")
    print(f"  - Fraud rate when present: {fraud_rate*100:.1f}%")

Signal correlation with fraud label:


payment_request:
  - Found in 3211 postings (22.4%)
  - Fraud rate when present: 3.4%

urgency:
  - Found in 1387 postings (9.7%)
  - Fraud rate when present: 5.5%

offplatform_contact:
  - Found in 356 postings (2.5%)
  - Fraud rate when present: 1.1%

vague_company:
  - Found in 2687 postings (18.8%)
  - Fraud rate when present: 18.0%

salary_anomaly:
  - Found in 56 postings (0.4%)
  - Fraud rate when present: 19.6%


In [4]:
# Show fraudulent examples with high signal counts
train_with_signals['total_signals'] = signals_df.sum(axis=1)

print("FRAUDULENT POSTINGS WITH MOST RED FLAGS:")
print("="*80)

fraud_examples = train_with_signals[train_with_signals['fraudulent'] == 1].nlargest(5, 'total_signals')

for idx, row in fraud_examples.iterrows():
    print(f"\nRed flags: {int(row['total_signals'])}/5")
    print(f"Title: {row['title']}")
    print(f"Signals: payment={row['payment_request']}, urgency={row['urgency']}, "
          f"offplatform={row['offplatform_contact']}, vague_company={row['vague_company']}, "
          f"salary_anomaly={row['salary_anomaly']}")
    print(f"Description: {row['description'][:200]}...")
    print("-"*80)

FRAUDULENT POSTINGS WITH MOST RED FLAGS:

Red flags: 4/5
Title: Webcam Model
Signals: payment=1, urgency=1, offplatform=1, vague_company=1, salary_anomaly=0
Description: Internet Modeling is a premier adult modeling agency recruiting and hiring webcam models for high paying webcam jobs. We are one of the oldest and most experienced online modeling agencies, representi...
--------------------------------------------------------------------------------

Red flags: 3/5
Title: Part Time Cash Jobs To Immediate Starts.
Signals: payment=1, urgency=1, offplatform=0, vague_company=1, salary_anomaly=0
Description: Part Time Cash Jobs To Immediate Starts.No Experience Required And Never Any Fees.Work Anytime 1 To 2 Hrs Daily In Free Time.Earn Easily $400 To $500 Extra Per Day.Totally Free To Join &amp; Suitable ...
--------------------------------------------------------------------------------

Red flags: 3/5
Title: Urgent Cash Jobs, Part Time & Full Time.
Signals: payment=1, urgency=1, offplatf

In [6]:
# Save signals for later use
train_with_signals.to_csv('C:/Users/diyad/Desktop/HireSafe/data/train_with_signals.csv', index=False)

print("✓ Saved train_with_signals.csv")
print(f"\nDataset now has {len(train_with_signals.columns)} columns including:")
print("  - Original features")
print("  - 5 extracted signal features")

✓ Saved train_with_signals.csv

Dataset now has 26 columns including:
  - Original features
  - 5 extracted signal features
